[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/bending/bending.ipynb)

# Unsymmetric Bending

Hang a weight from the end of a Z-shaped bracket, and its tip moves sideways as well as down. Why does the shape of the section change the direction of bending? We will build its response by adding the pulls of its fibres, using lines and motions in projective geometric algebra.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline

import numpy as np
from IPython.display import Image, display

from numga import NumpyContext, concatenate
from numga.algebras import PGA3D
from examples.animation import save_animation, save_figure
from examples.mechanics.bending import render
from examples.mechanics.bending.core import Beam, Pushes, direction, point
from examples.mechanics.bending.mesh import NAMES, section_meshes
from examples.mechanics.simplex import barycentric_samples, simplex_inertia_weights

In [ ]:
ga = PGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Point = ga.gatype.antivector()
Line = ga.gatype.bivector()
Motor = ga.gatype.rotor()
Strain = ga.gatype(ga.subspace("wz yz zx"))                # lines in the section's plane
Resultant = ga.gatype(ga.subspace("xy wx wy"))             # lines along the beam, and couples across it
Stiffness = ga.gatype((Resultant, Strain))                 # Resultant <- Strain: strain to the fibres' pull

section = mv.z                                             # [] Plane: the cross-section at the clamp
along = mv.z.dual()                                        # [] Point: the point at infinity along the beam
slide = section ^ mv.w                                     # [] Line: the twist that carries a section along the beam
across, up = direction([1.0, 0.0]), direction([0.0, 1.0])  # points at infinity across the section, and up

In [ ]:
sections = section_meshes()
save_figure(render.draw_meshes(sections, NAMES), "bending_meshes");

## 1. Each fibre contributes a pull

A beam bends because its fibres stretch on one side and shorten on the other. In a cross-section, that stretch varies linearly and vanishes along a line: the neutral axis. PGA lets us use this line, with its magnitude retained, as the strain itself. A fibre is another line, `along & point`; the scalar pairing `strain & fibre` measures its stretch and vanishes when the fibre meets the neutral axis. Leaving `Strain` open turns that measurement into a linear map. Multiplying it by the fibre gives the pull that fibre contributes, and summing gives the section's stiffness.

Three samples per triangle integrate the fibre dyads exactly, using the simplex second-moment rule. The samples carry square roots of their integration areas, so their two appearances in each dyad supply the area once.

In [ ]:
# The same material fills every triangle.
MODULUS = 1.0

In [ ]:
def areas(triangles: Point) -> Scalar:
    """Each triangle's area, from the plane through its three corners."""
    return (triangles[..., 0] & triangles[..., 1] & triangles[..., 2]).norm() / 2


def centroid(triangles: Point) -> Point:
    """Each triangle contributes its area times the mean of its corners."""
    return (areas(triangles)[..., None] * triangles / 3).sum(axis=(-2, -1)).normalized()


def stiffness(triangles: Point, modulus: float) -> Stiffness:
    """Sum the fibres' pulls under a strain, integrating each triangle exactly."""
    weights = simplex_inertia_weights(3)
    samples = barycentric_samples(weights, triangles) * areas(triangles).square_root()[..., None]
    fibres = along & samples                                               # [triangles, samples] Resultant
    stretches = Strain & fibres                                             # [triangles, samples] Scalar <- Strain
    # Each point appears twice, so its square-root area supplies one area in this dyad.
    tensions = modulus * fibres * stretches                                 # [triangles, samples] Resultant <- Strain
    return tensions.sum(axis=(-2, -1))                                      # [] Resultant <- Strain


triangles, boundary = sections[NAMES.index("Z-section")]
stiffness(triangles, MODULUS)

The sum already contains the area and both the first and second moments of the section: they are consequences of adding the fibre responses. There is no need to find a centroid or introduce lever arms before constructing the stiffness. Its input and output are different families of lines: `Strain` contains lines in the section plane, while `Resultant` contains axial forces and bending couples. The regressive pairing connects those families even though the PGA metric is degenerate.

## 2. From the load to the bent beam

The tip load is a line through its application point. Meeting it with a section plane and joining the result with `along` gives the axial force and bending couple carried by the fibres there. Solving the stiffness gives the strain, whose line is also the neutral axis we draw. That strain adds a turn to the unstrained slide along the beam; composing the motors carries us from the clamp to the tip. The load is evaluated on the straight beam, so this is a small-deflection load calculation followed by a finite-motion reconstruction, not a nonlinear equilibrium solve.

The beam is assumed slender, but the cross-section need not be thin-walled: a solid circle also fits this construction. Twist about the beam axis is absent from `Strain`: the fibre sum measures axial stretch, not shear. Torsional stiffness generally needs a separate cross-section field solve for warping or a stress function; it does not follow from this sum. For an open section such as the slotted tube, a transverse load can produce torsion as well; the drawing shows only the bending response. The kern concerns axial compression and bending.

In [ ]:
# The beam's length, the pieces it is cut into, and the load at its tip.
LENGTH = 10.0
PIECES = 40
LOAD = 4e-5


In [ ]:
def carried(loads: Line, stations: np.ndarray) -> Resultant:
    """What the fibres carry at each station along the beam: the load seen from the station, met with
    the section's plane, and joined with the point at infinity along the beam."""
    shifts = (slide * (stations / 2)).exp()                                  # [stations] Motor
    return along & ((shifts << loads[..., None]) ^ section)                  # [..., stations] Resultant


def bend(strains: Strain, step: float) -> Motor:
    """The frame of each cross-section from the clamp to the tip `[..., pieces + 1]`."""
    # Integrate each constant strain exactly; this does not update the load on the bent geometry.
    twists = ((slide + strains) * (step / 2)).exp()                          # [..., pieces] Motor
    clamp = mv.rotor().broadcast_to(twists.shape[:-1] + (1,))                # [..., 1] Motor
    # Each later twist multiplies from the right: the reverse of the running product of the reverses.
    return concatenate([clamp, twists.reverse().cumprod(axis=-1).reverse()], axis=-1)   # [..., pieces + 1] Motor


def stress(strains: Strain, modulus: float, points: Point) -> Scalar:
    """The stress along the beam at points of the section: the modulus times each fibre's stretch."""
    return modulus * (strains & (along & points))                            # [...] Scalar


def cantilever(triangles: Point, boundary: Point, loads: Point) -> Beam:
    """A cantilever with its tip load through the centroid, batched over load directions."""
    step = LENGTH / PIECES
    stations = (np.arange(PIECES) + 0.5) * step                             # [PIECES]: the middle of each piece
    rigidity = stiffness(triangles, MODULUS)                            # [] Resultant <- Strain
    centre = centroid(triangles)                                       # [] Point
    tip = (slide * (LENGTH / 2)).exp() >> centre                      # [] Point: the tip of the straight beam
    # Lever arms are measured on the straight reference beam.
    strains = rigidity.solve(carried(tip & loads, stations))           # [..., PIECES] Strain
    frames = bend(strains, step)                                       # [..., PIECES + 1] Motor
    # Principal bending axes pass through the centroid; the metric does not see the axial slide.
    _, axes = (Strain | Strain).eigh(Strain & rigidity)                    # [3] Strain: weakest first, then the slide
    stresses = stress(strains[..., 0, None, None], MODULUS, triangles)     # [..., triangles, 3] Scalar
    return Beam(triangles, boundary, loads, strains, frames, frames >> centre, axes[:2], stresses)


downward = [cantilever(triangles, boundary, LOAD * -up) for triangles, boundary in sections]
save_figure(render.draw_beams(downward, NAMES), "bending_sections");

## 3. Turning the load

The strain's neutral axis passes through the centroid under a transverse load, but need not be perpendicular to that load. That is why the Z-section bends sideways. Turn its load through a full circle: along the principal bending directions, the tip follows the load; between them, it veers toward the more compliant direction. In the small-deflection limit the tip traces an ellipse. The motor reconstruction retains finite rotations, so its path only approaches that ellipse as the load decreases.

In [ ]:
FRAMES = 48
DURATION_MS = 60

angles = np.linspace(0.0, 2 * np.pi, FRAMES, endpoint=False)                 # [FRAMES]
loads = (mv.xy * (-angles / 2)).exp() >> (LOAD * -up)                         # [FRAMES] Point: at infinity
triangles, boundary = sections[NAMES.index("Z-section")]
turning = cantilever(triangles, boundary, loads)
display(Image(filename=save_animation(render.animate_turning(turning), "bending_turning", DURATION_MS)))

## 4. Where can we push without stretching any fibres?

Push along the beam through a point of the section. Its fibre line is the resultant, so the same stiffness solve gives its neutral axis. A push through the centroid compresses every fibre equally: the neutral axis is at infinity. Moving the push off-centre brings that line into finite space; when it crosses the section, some fibres go into tension. The allowed push points form the kern, also called the core of the section, shown in purple. The inverse stiffness thus associates each push point with a neutral line: a polarity within the section plane, with a direct mechanical meaning.

In the notation of classical strength of materials the neutral axis of a push at $(e_x, e_y)$ in principal axes reads $1 + \frac{e_x x}{i_y^2} + \frac{e_y y}{i_x^2} = 0$, with $i_x, i_y$ the radii of gyration: the antipolar of the push point with respect to the central ellipse of inertia.

In [ ]:
# The pushes along the beam: their size, and the points they go through, across and up, one inside
# every section's kern and one outside; the half width of the view of the kern and its grid.
PUSH = 1e-3
PUSH_POINTS = [[0.03, 0.04], [0.1, 0.3]]
REACH = 0.75
KERN_GRID = 161

In [ ]:
def pulls(stiffness: Stiffness, points: Point) -> Strain:
    """The strain of a unit pull along the beam through each point of the section: its neutral axis,
    the point's polar line."""
    return stiffness.solve(along & points)                                   # [...] Strain


def push_section(triangles: Point, points: Point, view: Point) -> Pushes:
    """The stresses of axial pushes, and the corner stretches defining the kern."""
    rigidity = stiffness(triangles, MODULUS)                            # [] Resultant <- Strain
    strains = -PUSH * pulls(rigidity, points)                          # [pushes] Strain
    stresses = stress(strains[..., None, None], MODULUS, triangles)       # [pushes, triangles, 3] Scalar
    pulled = pulls(rigidity, view)                                     # [rows, columns] Strain
    corners = stress(pulled[..., None, None], MODULUS, triangles)       # [rows, columns, triangles, 3] Scalar
    return Pushes(triangles, points, strains, stresses, view, corners)


points = point(PUSH_POINTS)
reach = np.linspace(-REACH, REACH, KERN_GRID)
view = point(np.stack(np.meshgrid(reach, reach), axis=-1))
pushes = [push_section(triangles, points, view) for triangles, boundary in sections]
save_figure(render.draw_pushes(pushes, NAMES), "bending_kern");

The purple region marks load positions, not material. Each corner requires its stress to stay compressive; because that stress varies linearly with the load position, each requirement bounds a half-plane. Their intersection is a polygon: a diamond for the I-section and a tilted polygon for the Z-section. The boundary is where a first corner reaches zero stress. The drawing samples these inequalities on a grid; the polygonal shape comes from the mechanics, not the grid.

The slotted tube shows that the kern need not lie in the material: a load resultant transmitted through an end plate can lie in the hollow. Removing the sector on the right moves the centroid left and changes the stiffness, so the kern loses the circular symmetry of a closed tube.

The same weighted line measures fibre stretch, locates zero stress, and generates the bending motion. The stiffness links that line to the forces through a sum of fibre dyads; solving it makes the neutral axis a polar line. Symmetry has a physical reading too: a pull through one point stretches the fibre through another by the same amount as the exchanged experiment. These connections are what the PGA formulation makes visible. The constitutive assumption still comes from the material: each fibre pulls in proportion to its stretch.